In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OPENAI_API_KEY is set.")
else:
    print("OPENAI_API_KEY is not set.")

OPENAI_API_KEY is set.


In [20]:
from langchain_openai import ChatOpenAI

In [21]:
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# **RAG IMPLEMENTATION WITH adding more PDFs, persist store locally**

#### **Step-1: Extracting Text from PDFs**

In [22]:
from langchain_community.document_loaders import PyPDFLoader
pdf_path = "./docs/fabric-admin.pdf"
loader = PyPDFLoader(pdf_path)

docs = loader.load()

docs

[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26056.01', 'creator': 'Microsoft Learn', 'creationdate': '2026-05-13T10:44:36+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2026-05-13T10:44:36+00:00', 'source': './docs/fabric-admin.pdf', 'total_pages': 307, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nFabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.26056.01', 'creator': 'M

#### **Creating/ modifying own metadata.**

In [30]:
for i in docs:
    i.metadata = {"source": f"{pdf_path}", "sign" : "Shan_Rag_102"}

#### **Step-2: Splitting the document in chunks.**

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = text_splitter.split_documents(docs)
chunks

[Document(metadata={'source': './docs/fabric-admin.pdf', 'sign': 'Shan_Rag_102'}, page_content='Tell us about your PDF experience.\nFabric administration documentation\nLearn about the Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is administration in Fabric?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration'),
 Document(metadata={'source': './docs/fabric-admin.pdf', 'sign': 'Shan_Rag_102'}, page_content='ｃHOW-TO GUIDE\nSet up git integration\nSet up item certification\nConfigure notifications\nSet up metadata scanning'),
 Document(metadata={'source': './docs/fabric-admin.pdf', 'sign': 'Shan_Rag_102'}, page_content='Enable content certification\nEnable service principal authentication\nConfigure

In [32]:
len(chunks)
# PyPDFLoader creates metadata for each chunk, which includes the source document and the page number
chunks[0].metadata

{'source': './docs/fabric-admin.pdf', 'sign': 'Shan_Rag_102'}

#### **Step-3: Creating Embeddings for the chunks.**

In [33]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    check_embedding_ctx_length=False,
)

#### **Step-4: Store Embeddings in the existing local Vector Store.**

In [34]:
from langchain_community.vectorstores import Chroma

vectorstore: Chroma = Chroma(embedding_function=embedding_model, persist_directory="./vectorstore")
vectorstore

#### **Step-5: Add chunks to the existing vector store.**

In [35]:
vectorstore.add_documents(chunks)

['5e34cb87-f880-4abd-9498-97194227d60e',
 '70794162-b811-4dad-91b8-51b0a534b2d8',
 'fff85d59-3e8f-4550-aad2-e00923eb5e0f',
 '0fd2874f-7a33-44bf-b0a0-0d4fa9a7d876',
 '0f3110e5-8aec-4a17-9ac7-516ad2cd6126',
 '0f00e310-eb2d-4299-bbfe-8d621865af43',
 '8bc1702e-b2f6-4510-ac85-49a1fe088bb6',
 '02fa40a5-35c3-4d7c-9138-56dc5acca6f8',
 '766bd52a-01c9-44a2-8beb-9e2675aeabd9',
 'fc8eebdb-7c84-46e1-9032-da34eded8cca',
 '5451b537-25d9-4a71-b4d4-01baf4a8ecd6',
 'f1a6dec7-5d60-4697-93ab-ea94cc5ef657',
 '91cb3342-9721-4462-bdf7-1aee52c71fa9',
 'c6e3bc95-c771-4163-8472-4a92000e7996',
 '844633e9-1a14-4ead-b877-71b831bf55e0',
 'f658fdc3-404a-4466-a768-7ddd39d87c84',
 'df596751-3318-42ac-93a9-34778da08328',
 '935317d7-f699-4912-a35d-f3594cf33854',
 '784e41f7-865d-482a-870f-fb018dbfbc5f',
 '8cb11699-0af9-4b8b-9826-86feafdbd5da',
 '0afd9610-174f-4a44-819f-6b8331d3d378',
 'e658a551-007a-42b2-8a69-fcbd0b8d796d',
 '31b31fb3-36b1-4f4e-94ee-897a64b7b634',
 'cffda671-1250-4947-b2e3-26da95d39b2b',
 'f1ad69d5-d1fd-

#### **Step-6: Semantic Search.**

In [37]:
vectorstore.similarity_search(query="What is Fabric admin", k=2)

[Document(metadata={'source': './docs/fabric-admin.pdf - page 2', 'sign': 'Shan_Rag_101'}, page_content='policies for data access, sharing, classification, and auditing.\nThis article gives an overview of Fabric admin responsibilities, tools, and key tasks for managing\nyour Fabric environment.\nFabric admins use a combination of the Fabric admin portal and related admin tools to perform\ntasks based on their areas of responsibility. The following are common admin tasks and the tools\ntypically used for each one.\nMicrosoft Fabric admin portal\nAcquire and work with capacities\nEnsure quality of service'),
 Document(metadata={'sign': 'Shan_Rag_102', 'source': './docs/fabric-admin.pdf'}, page_content='policies for data access, sharing, classification, and auditing.\nThis article gives an overview of Fabric admin responsibilities, tools, and key tasks for managing\nyour Fabric environment.\nFabric admins use a combination of the Fabric admin portal and related admin tools to perform\ntas